In [3]:
import numpy as np

In [ ]:

# ------------------------------------------
# Call the function (we'll enable prints by
# temporarily adding them to the source)
# ------------------------------------------

# For exploration, I'll create a temporary copy of the function
# with print statements enabled inside.

def get_labels_resorting_array_with_prints(
    types: np.ndarray,
    shapes: np.ndarray,
    shapes_inv: np.ndarray = None,
    transpose_neg: bool = False,
) -> np.ndarray:
    print("Starting get_labels_resorting_array_with_prints...")
    print(f"Input types: {types}")
    print(f"Input shapes:\n{shapes}")
    if shapes_inv is not None:
        print(f"Input shapes_inv:\n{shapes_inv}")
    print(f"Transpose negative: {transpose_neg}")
    n_entries = types.shape[0]

    if shapes_inv is None:
        shapes_inv = shapes
    

    n_types = shapes.shape[1]*2-1 # take into account shapes and shapes_inv
    ntypes_int = shapes.shape[1] # this is the number of integers >=0 we have.
    # types_order = np.zeros(n_types, dtype=np.int64)
    # i = 1
    # for typ in range(1, ntypes_int):
    #     types_order[i] = typ
    #     types_order[i+1] = -typ
    #     i += 2
    print(f"Calculated n_types: {n_types}")
    if (shapes[:, 0] != shapes_inv[:, 0]).any():
        raise ValueError("For nodes, they must have same shape, so shapes[0] == shapes_inv[0]")
    sizes = np.zeros(n_types, dtype=np.int64)
    for typ in range(shapes.shape[1]):
        sizes[typ] = shapes[0, typ] * shapes[1, typ]
        sizes[-typ] = shapes_inv[0, typ] * shapes_inv[1, typ]  # the inverse

    # Count the number of entries of each type
    type_nlabels = np.zeros(n_types, dtype=np.int64)
    for i_edge in range(n_entries):
        typ = types[i_edge]
        type_nlabels[typ] += sizes[typ] # in the ex: 4 of type 1 (size 5) 4x5 - 20.
        print(f"Edge {i_edge}: type {typ}, size {sizes[typ]}")
    offset = np.zeros(n_types, dtype=np.int64)
    prev_type = 0
    for typ in range(1, ntypes_int):  # Here we just range in n_types, but this takes the negatives
        offset[typ] = offset[prev_type] + type_nlabels[prev_type]
        offset[-typ] = offset[typ] + type_nlabels[typ]  # < 0
        prev_type = -typ  # we have to continue from the negative type, because the next positive type will be after it.
    # offset goes in order 1 -1 2 -2...
    # for typ in range(1, n_types):
    #     prev_typ = types_order[typ - 1]
    #     offset[typ] = offset[prev_typ] + type_nlabels[prev_typ]

    # BORRAR
    print(f"las taken type = {prev_type}, offset[prev_type] = {offset[prev_type]}, type_nlabels[prev_type] = {type_nlabels[prev_type]}")
    print(f"offset[-ntypes_int] = {offset[-ntypes_int]}, type_nlabels[-ntypes_int] = {type_nlabels[-ntypes_int]}")

    total_len = offset[-ntypes_int] + type_nlabels[-ntypes_int]
    print(f"Total length of indices array: {total_len}")
    indices = np.empty(total_len, dtype=np.int64)

    type_i = np.zeros_like(sizes)
    i = 0

    print("=" * 60)
    print("Initial state:")
    print(f"  n_entries = {n_entries}, n_types = {n_types}")
    print(f"  sizes      = {sizes}")
    print(f"  type_nlabels = {type_nlabels}")
    print(f"  offset     = {offset}")
    print(f"  total_len  = {total_len}")
    print("=" * 60)

    for i_edge in range(n_entries):
        typ = types[i_edge]
        abs_type = abs(typ)
        block_size = sizes[typ]
        start = offset[typ] + type_i[typ]

        print(f"\n--- Edge {i_edge}: type={typ}, abs_type={abs_type}, "
              f"block_size={block_size}, start={start} ---")

        if transpose_neg and typ < 0:
            cols, rows = shapes[0, abs_type], shapes[1, abs_type]
            print(f"  Transposing: original shape ({shapes[0, abs_type]}x{shapes[1, abs_type]}) "
                  f"=> new dims ({rows}x{cols})")
            for jrow in range(rows):
                for jcol in range(cols):
                    idx_val = start + jcol * rows + jrow
                    indices[i] = idx_val
                    print(f"    indices[{i}] = {idx_val}  (jrow={jrow}, jcol={jcol})")
                    i += 1
        else:
            print(f"  Normal (no transpose): filling indices[{i} : {i+block_size}] "
                  f"with {start} ... {start+block_size-1}")
            for j in range(start, start + block_size):
                print(f"    indices[{i}] = {j}")
                indices[i] = j
                i += 1

        type_i[typ] += block_size
        print(f"  Updated type_i[{abs_type}] = {type_i[abs_type]}")

    print("\n" + "=" * 60)
    print("Final indices:", indices)
    print("=" * 60)
    return indices

In [9]:


# Example data: 
# for the case 
# point_1 = PointBasis("A", R=2, basis="0e", basis_convention="spherical", matrix_role='row')  # "0e"
# point_2 = PointBasis("A", R=2, basis="2x0e", basis_convention="spherical", matrix_role='col')
# point_3 = PointBasis("B", R=5, basis="0e + 1o", basis_convention="spherical", matrix_role='row')
# point_4 = PointBasis("B", R=5, basis="2x0e + 1o", basis_convention="spherical", matrix_role='col')

# types:  [ 1 -1  1 -1]
# shapes:  [[1 1 4 4]
#  [2 5 2 5]]
# transpose_neg:  False
shapes = np.array([
    [1, 1, 4],   # rows per type
    [2, 5, 5]    # cols per type
], dtype=np.int64)

# Edge order: type 0, type 1, then type 1 but transposed
types = np.array([1, -1, 1, -1], dtype=np.int64)




# ------------------------------------------
# Run the exploration
# ------------------------------------------
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    transpose_neg=True
)


# # Apply to the labels to see the reordered result
# sorted_labels = labels[indices]

# print("\nOriginal type‑grouped labels (flat):")
# print(labels)
# print("\nResorting indices:")
# print(indices)
# print("\nLabels after reordering (edge‑wise order):")
# print(sorted_labels)

Starting get_labels_resorting_array_with_prints...
Input types: [ 1 -1  1 -1]
Input shapes:
[[1 1 4]
 [2 5 5]]
Transpose negative: True
Calculated n_types: 5
Edge 0: type 1, size 5
Edge 1: type -1, size 5
Edge 2: type 1, size 5
Edge 3: type -1, size 5
Total length of indices array: 20
Initial state:
  n_entries = 4, n_types = 5
  sizes      = [ 2  5 20 20  5]
  type_nlabels = [ 0 10  0  0 10]
  offset     = [ 0  0 20 20 10]
  total_len  = 20

--- Edge 0: type=1, abs_type=1, block_size=5, start=0 ---
  Normal (no transpose): filling indices[0 : 5] with 0 ... 4
    indices[0] = 0
    indices[1] = 1
    indices[2] = 2
    indices[3] = 3
    indices[4] = 4
  Updated type_i[1] = 5

--- Edge 1: type=-1, abs_type=1, block_size=5, start=10 ---
  Transposing: original shape (1x5) => new dims (5x1)
    indices[5] = 10  (jrow=0, jcol=0)
    indices[6] = 11  (jrow=1, jcol=0)
    indices[7] = 12  (jrow=2, jcol=0)
    indices[8] = 13  (jrow=3, jcol=0)
    indices[9] = 14  (jrow=4, jcol=0)
  Updated 

In [10]:
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    transpose_neg=False
)

Starting get_labels_resorting_array_with_prints...
Input types: [ 1 -1  1 -1]
Input shapes:
[[1 1 4]
 [2 5 5]]
Transpose negative: False
Calculated n_types: 5
Edge 0: type 1, size 5
Edge 1: type -1, size 5
Edge 2: type 1, size 5
Edge 3: type -1, size 5
Total length of indices array: 20
Initial state:
  n_entries = 4, n_types = 5
  sizes      = [ 2  5 20 20  5]
  type_nlabels = [ 0 10  0  0 10]
  offset     = [ 0  0 20 20 10]
  total_len  = 20

--- Edge 0: type=1, abs_type=1, block_size=5, start=0 ---
  Normal (no transpose): filling indices[0 : 5] with 0 ... 4
    indices[0] = 0
    indices[1] = 1
    indices[2] = 2
    indices[3] = 3
    indices[4] = 4
  Updated type_i[1] = 5

--- Edge 1: type=-1, abs_type=1, block_size=5, start=10 ---
  Normal (no transpose): filling indices[5 : 10] with 10 ... 14
    indices[5] = 10
    indices[6] = 11
    indices[7] = 12
    indices[8] = 13
    indices[9] = 14
  Updated type_i[1] = 5

--- Edge 2: type=1, abs_type=1, block_size=5, start=5 ---
  Norm

In [20]:
shapes = np.array([
    [1, 1, 4],   # rows per type
    [2, 5, 5]    # cols per type
], dtype=np.int64)

# Edge order: type 0, type 1, then type 1 but transposed
types = np.array([1, -1, 1, -1, 2, -2], dtype=np.int64)




# ------------------------------------------
# Run the exploration
# ------------------------------------------
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    transpose_neg=True
)
print("\nResorting indices with transpose_neg=True:")
print(indices)

Starting get_labels_resorting_array_with_prints...
Input types: [ 1 -1  1 -1  2 -2]
Input shapes:
[[1 1 4]
 [2 5 5]]
Transpose negative: True
Calculated n_types: 5
Edge 0: type 1, size 5
Edge 1: type -1, size 5
Edge 2: type 1, size 5
Edge 3: type -1, size 5
Edge 4: type 2, size 20
Edge 5: type -2, size 20
Total length of indices array: 60
Initial state:
  n_entries = 6, n_types = 5
  sizes      = [ 2  5 20 20  5]
  type_nlabels = [ 0 10 20 20 10]
  offset     = [ 0  0 10 30 50]
  total_len  = 60

--- Edge 0: type=1, abs_type=1, block_size=5, start=0 ---
  Normal (no transpose): filling indices[0 : 5] with 0 ... 4
    indices[0] = 0
    indices[1] = 1
    indices[2] = 2
    indices[3] = 3
    indices[4] = 4
  Updated type_i[1] = 5

--- Edge 1: type=-1, abs_type=1, block_size=5, start=50 ---
  Transposing: original shape (1x5) => new dims (5x1)
    indices[5] = 50  (jrow=0, jcol=0)
    indices[6] = 51  (jrow=1, jcol=0)
    indices[7] = 52  (jrow=2, jcol=0)
    indices[8] = 53  (jrow=3, j

In [11]:
shapes_inv = np.array([
    [1, 4, 4],   # rows per type
    [2, 2, 5]    # cols per type
], dtype=np.int64)
shapes = np.array([
    [1, 1, 4],   # rows per type
    [2, 5, 5]    # cols per type
], dtype=np.int64)
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    shapes_inv=shapes_inv,
    transpose_neg=False
)

Starting get_labels_resorting_array_with_prints...
Input types: [ 1 -1  1 -1]
Input shapes:
[[1 1 4]
 [2 5 5]]
Input shapes_inv:
[[1 4 4]
 [2 2 5]]
Transpose negative: False
Calculated n_types: 5
Edge 0: type 1, size 5
Edge 1: type -1, size 8
Edge 2: type 1, size 5
Edge 3: type -1, size 8
Total length of indices array: 26
Initial state:
  n_entries = 4, n_types = 5
  sizes      = [ 2  5 20 20  8]
  type_nlabels = [ 0 10  0  0 16]
  offset     = [ 0  0 26 26 10]
  total_len  = 26

--- Edge 0: type=1, abs_type=1, block_size=5, start=0 ---
  Normal (no transpose): filling indices[0 : 5] with 0 ... 4
    indices[0] = 0
    indices[1] = 1
    indices[2] = 2
    indices[3] = 3
    indices[4] = 4
  Updated type_i[1] = 5

--- Edge 1: type=-1, abs_type=1, block_size=8, start=10 ---
  Normal (no transpose): filling indices[5 : 13] with 10 ... 17
    indices[5] = 10
    indices[6] = 11
    indices[7] = 12
    indices[8] = 13
    indices[9] = 14
    indices[10] = 15
    indices[11] = 16
    indice

In [12]:
types = np.array([0, 1, 0], dtype=np.int64)
shapes = np.array([
    [1, 4],   # rows per type
    [2, 5]    # cols per type
], dtype=np.int64)
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    transpose_neg=False
)

Starting get_labels_resorting_array_with_prints...
Input types: [0 1 0]
Input shapes:
[[1 4]
 [2 5]]
Transpose negative: False
Calculated n_types: 3
Edge 0: type 0, size 2
Edge 1: type 1, size 20
Edge 2: type 0, size 2
Total length of indices array: 24
Initial state:
  n_entries = 3, n_types = 3
  sizes      = [ 2 20 20]
  type_nlabels = [ 4 20  0]
  offset     = [ 0  4 24]
  total_len  = 24

--- Edge 0: type=0, abs_type=0, block_size=2, start=0 ---
  Normal (no transpose): filling indices[0 : 2] with 0 ... 1
    indices[0] = 0
    indices[1] = 1
  Updated type_i[0] = 2

--- Edge 1: type=1, abs_type=1, block_size=20, start=4 ---
  Normal (no transpose): filling indices[2 : 22] with 4 ... 23
    indices[2] = 4
    indices[3] = 5
    indices[4] = 6
    indices[5] = 7
    indices[6] = 8
    indices[7] = 9
    indices[8] = 10
    indices[9] = 11
    indices[10] = 12
    indices[11] = 13
    indices[12] = 14
    indices[13] = 15
    indices[14] = 16
    indices[15] = 17
    indices[16] = 18

In [8]:
types = np.array([1, -1, 1, -1, 1, -1, 1, -1], dtype=np.int64)
shapes = np.array([
    [1, 1, 4],   # rows per type
    [2, 5, 5]    # cols per type
], dtype=np.int64)
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    transpose_neg=False
)

Starting get_labels_resorting_array_with_prints...
Input types: [ 1 -1  1 -1  1 -1  1 -1]
Input shapes:
[[1 1 4]
 [2 5 5]]
Transpose negative: False
Initial state:
  n_entries = 8, n_types = 3
  sizes      = [ 2  5 20]
  sizes_inv  = [ 2  5 20]
  type_nlabels = [ 0 40  0]
  offset     = [ 0  0 40]
  total_len  = 40

--- Edge 0: type=1, abs_type=1, block_size=5, start=0 ---
  Normal (no transpose): filling indices[0 : 5] with 0 ... 4
    indices[0] = 0
    indices[1] = 1
    indices[2] = 2
    indices[3] = 3
    indices[4] = 4
  Updated type_i[1] = 5

--- Edge 1: type=-1, abs_type=1, block_size=5, start=5 ---
  Negative type (no transpose): filling indices[5 : 10] with 5 ... 9
    indices[5] = 5
    indices[6] = 6
    indices[7] = 7
    indices[8] = 8
    indices[9] = 9
  Updated type_i[1] = 10

--- Edge 2: type=1, abs_type=1, block_size=5, start=10 ---
  Normal (no transpose): filling indices[10 : 15] with 10 ... 14
    indices[10] = 10
    indices[11] = 11
    indices[12] = 12
    ind

In [7]:
types = np.array([1, -1, 1, -1, 1, -1, 1, -1], dtype=np.int64)
shapes_inv = np.array([
    [1, 4, 4],   # rows per type
    [2, 2, 5]    # cols per type
], dtype=np.int64)
shapes = np.array([
    [1, 1, 4],   # rows per type
    [2, 5, 5]    # cols per type
], dtype=np.int64)
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    shapes_inv=shapes_inv,
    transpose_neg=False
)

Starting get_labels_resorting_array_with_prints...
Input types: [ 1 -1  1 -1  1 -1  1 -1]
Input shapes:
[[1 1 4]
 [2 5 5]]
Input shapes_inv:
[[1 4 4]
 [2 2 5]]
Transpose negative: False
Initial state:
  n_entries = 8, n_types = 3
  sizes      = [ 2  5 20]
  sizes_inv  = [ 2  8 20]
  type_nlabels = [ 0 52  0]
  offset     = [ 0  0 52]
  total_len  = 52

--- Edge 0: type=1, abs_type=1, block_size=5, start=0 ---
  Normal (no transpose): filling indices[0 : 5] with 0 ... 4
    indices[0] = 0
    indices[1] = 1
    indices[2] = 2
    indices[3] = 3
    indices[4] = 4
  Updated type_i[1] = 5

--- Edge 1: type=-1, abs_type=1, block_size=5, start=5 ---
  Negative type (no transpose): filling indices[5 : 13] with 5 ... 12
    indices[5] = 5
    indices[6] = 6
    indices[7] = 7
    indices[8] = 8
    indices[9] = 9
    indices[10] = 10
    indices[11] = 11
    indices[12] = 12
  Updated type_i[1] = 13

--- Edge 2: type=1, abs_type=1, block_size=5, start=13 ---
  Normal (no transpose): filling i

In [ ]:

# ------------------------------------------
# Call the function (we'll enable prints by
# temporarily adding them to the source)
# ------------------------------------------

# For exploration, I'll create a temporary copy of the function
# with print statements enabled inside.

def get_labels_resorting_array_with_prints_Sara_old_withinv(
    types: np.ndarray,
    shapes: np.ndarray,
    shapes_inv: np.ndarray = None,
    transpose_neg: bool = False,
) -> np.ndarray:
    print("Starting get_labels_resorting_array_with_prints...")
    print(f"Input types: {types}")
    print(f"Input shapes:\n{shapes}")
    if shapes_inv is not None:
        print(f"Input shapes_inv:\n{shapes_inv}")
    print(f"Transpose negative: {transpose_neg}")
    n_entries = types.shape[0]
    n_types = shapes.shape[1]
    if shapes_inv is None:
        shapes_inv = shapes
    sizes = np.zeros(n_types, dtype=np.int64)
    sizes_inv = np.zeros(n_types, dtype=np.int64)
    for typ in range(n_types):
        sizes[typ] = shapes[0, typ] * shapes[1, typ]
        sizes_inv[typ] = shapes_inv[0, typ] * shapes_inv[1, typ]

    # Count the number of entries of each type
    type_nlabels = np.zeros(n_types, dtype=np.int64)
    for i_edge in range(n_entries):
        if types[i_edge] > 0:
            typ = types[i_edge]
            type_nlabels[typ] += sizes[typ] # in the ex: 4 of type 1 (size 5) 4x5 - 20.
        else:
            typ = abs(types[i_edge])
            type_nlabels[typ] += sizes_inv[typ] # in the ex: 4 of type 1 (size 5) 4x5 - 20.
    offset = np.zeros(n_types, dtype=np.int64)
    for typ in range(1, n_types):
        offset[typ] = offset[typ - 1] + type_nlabels[typ - 1]

    total_len = offset[n_types - 1] + type_nlabels[n_types - 1]
    indices = np.empty(total_len, dtype=np.int64)

    type_i = np.zeros_like(sizes)
    i = 0

    print("=" * 60)
    print("Initial state:")
    print(f"  n_entries = {n_entries}, n_types = {n_types}")
    print(f"  sizes      = {sizes}")
    print(f"  sizes_inv  = {sizes_inv}")
    print(f"  type_nlabels = {type_nlabels}")
    print(f"  offset     = {offset}")
    print(f"  total_len  = {total_len}")
    print("=" * 60)

    for i_edge in range(n_entries):
        typ = types[i_edge]
        abs_type = abs(typ)
        block_size = sizes[abs_type]
        start = offset[abs_type] + type_i[abs_type]

        print(f"\n--- Edge {i_edge}: type={typ}, abs_type={abs_type}, "
              f"block_size={block_size}, start={start} ---")

        if transpose_neg and typ < 0:
            cols, rows = shapes[0, abs_type], shapes[1, abs_type]
            print(f"  Transposing: original shape ({shapes[0, abs_type]}x{shapes[1, abs_type]}) "
                  f"=> new dims ({rows}x{cols})")
            for jrow in range(rows):
                for jcol in range(cols):
                    idx_val = start + jcol * rows + jrow
                    indices[i] = idx_val
                    print(f"    indices[{i}] = {idx_val}  (jrow={jrow}, jcol={jcol})")
                    i += 1
        elif typ < 0:
            block_size = sizes_inv[abs_type]
            print(f"  Negative type (no transpose): filling indices[{i} : {i+block_size}] "
                  f"with {start} ... {start+block_size-1}")
            for j in range(start, start + block_size):
                indices[i] = j
                print(f"    indices[{i}] = {j}")
                i += 1
        else:
            block_size = sizes[abs_type]
            print(f"  Normal (no transpose): filling indices[{i} : {i+block_size}] "
                  f"with {start} ... {start+block_size-1}")
            for j in range(start, start + block_size):
                print(f"    indices[{i}] = {j}")
                indices[i] = j
                i += 1

        type_i[abs_type] += block_size
        print(f"  Updated type_i[{abs_type}] = {type_i[abs_type]}")

    print("\n" + "=" * 60)
    print("Final indices:", indices)
    print("=" * 60)
    return indices

In [ ]:
import numpy as np

import cython


@cython.boundscheck(False)
@cython.wraparound(False)
def get_labels_resorting_array(
    types: cython.integral[:],
    shapes: cython.integral[:, :],
    shapes_inv: cython.integral[:, :] = None, # SN: for cases where is not square is needed: the shape of edge i ! = -i
    transpose_neg: cython.bint = False,
):
    """
    The problem this function solves is that graph2mat executes edge/node operations
    per edge/node type. In the case where there are 3 types, for example, you end up with
    three arrays:

    labels_0 = [...] (n_0, x_0, y_0)
    labels_1 = [...] (n_1, x_1, y_1)
    labels_2 = [...] (n_2, x_2, y_2)

    Where n_i is the number of edges/nodes of type i, and x_i, y_i are the number of rows
    and columns of the block of type i.

    Since each type has a different block shape, they are always
    raveled and concatenated into a single array:

    labels = np.concatenate([labels_0.ravel(), labels_1.ravel(), labels_2.ravel()])

    The labels array is of course not in the same order as the target labels.

    This function receives the original order of types and then returns the indices
    to apply to the labels array to get the correct order. I.e.:

    sorted_labels = labels[indices]

    Extra complication for edges
    -----------------------------

    An interesting fact is that when grouping the edges by size, there might be edges
    in one direction and edges in the other. E.g.:

    Original basis: A, B, C

    where shape A == shape C != shape B. Then when grouping by size, we have:

    Basis: B, (A, C)

    In the target, you will always have AB and BC edges (never BA or CB). But once you
    group, you have to face the problem that now the order is B(A, C), and therefore
    AB edges have been reversed. That is, the predicted blocks are the transpose of
    the target blocks.

    I think this is only a problem for symmetric matrices where only
    one direction is predicted.
    """
    n_entries = types.shape[0]
    n_types: cython.int = shapes.shape[1]

    type: cython.int
    rows: cython.int
    cols: cython.int
    jrow: cython.int
    jcol: cython.int

    type_nlabels: cython.long[:] = np.zeros(n_types, dtype=int)
    offset: cython.long[:] = np.zeros(n_types, dtype=int)

    if shapes_inv is None:
        shapes_inv = shapes

    # Compute the sizes for each type
    sizes: cython.int[:] = np.zeros(n_types, dtype=np.int32)
    sizes_inv: cython.int[:] = np.zeros(n_types, dtype=np.int32)
    for type in range(n_types):
        sizes[type] = shapes[0, type] * shapes[1, type]
        sizes_inv[type] = shapes_inv[0, type] * shapes_inv[1, type]


    # Count the number of entries of each type
    for i_edge in range(n_entries):
        type: cython.int = types[i_edge]
        if type < 0:
            type_nlabels[abs(type)] += sizes_inv[abs(type)]
        else:
            type_nlabels[type] += sizes[type]

    # Cumsum of type_nlabels to understand where do the labels for
    # each type start.
    for type in range(1, n_types):
        offset[type] = offset[type - 1] + type_nlabels[type - 1]

    # Initialize the indices array.
    # (for each label value, index of the unsorted array where it is located)
    indices: cython.long[:] = np.empty(
        offset[n_types - 1] + type_nlabels[n_types - 1], dtype=int
    )

    type_i: cython.long[:] = np.zeros_like(sizes, dtype=int)
    i: cython.int = 0

    for i_edge in range(n_entries):
        type = types[i_edge]
        abs_type: cython.int = abs(type)

        block_size: cython.int = sizes[abs_type]
        start: cython.int = offset[abs_type] + type_i[abs_type]

        if transpose_neg and type < 0:
            # Get the transposed shape
            cols, rows = shapes[0, abs_type], shapes[1, abs_type]
            for jrow in range(rows):
                for jcol in range(cols):
                    indices[i] = start + jcol * rows + jrow
                    i += 1
        elif type < 0:
            block_size = sizes_inv[abs_type]
            for j in range(start, start + block_size):
                indices[i] = j
                i += 1
        else:
            for j in range(start, start + block_size):
                indices[i] = j
                i += 1

        type_i[abs_type] += block_size

    return np.asarray(indices)